# 08 - Model: Weighted Similarity

# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/processed/featured_dataset.csv')
df_full_standard = pd.read_csv('../data/processed/featured_full_standard.csv')
df_audio_standard = pd.read_csv('../data/processed/featured_audio_standard.csv')

with open('../config/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

audio_features = feature_sets['audio_features']
full_features = feature_sets['full_features']

print('shape:', df.shape)
print('audio features:', len(audio_features))
print('full features:', len(full_features))

shape: (88167, 32)
audio features: 11
full features: 21


# Weighted Similarity Function

In [2]:
def recommend_weighted(mood, df, scaled_df, features, weights, n=10, genre=None):
    # mood to feature targets
    mood_targets = {
        'happy': {'valence': 1.0, 'energy': 1.0},
        'angry': {'valence': 0.0, 'energy': 1.0},
        'sad':   {'valence': 0.0, 'energy': 0.0},
        'calm':  {'valence': 1.0, 'energy': 0.0},
    }
    
    if mood not in mood_targets:
        print(f'Invalid mood. Choose from: {list(mood_targets.keys())}')
        return None
    
    target = mood_targets[mood]
    print(f'Mood: {mood}')
    print(f'Target → valence: {target["valence"]}, energy: {target["energy"]}')
    print(f'Weights: {weights}')
    print()
    
    # apply genre filter if specified
    if genre:
        mask = df['track_genre'].str.contains(genre, case=False)
        filtered_df = df[mask]
        filtered_scaled = scaled_df[mask]
    else:
        filtered_df = df
        filtered_scaled = scaled_df
    
    # filter by mood category
    filtered_df = filtered_df[filtered_df['mood'] == mood]
    filtered_scaled = filtered_scaled.loc[filtered_df.index]
    
    print(f'Songs in mood pool: {len(filtered_df)}')
    
    # create weight vector
    weight_vector = np.array([weights.get(f, 1.0) for f in features])
    
    # weighted feature matrix
    weighted_matrix = filtered_scaled[features].values * weight_vector
    
    # create target vector from mood targets
    target_vector = np.zeros((1, len(features)))
    for i, f in enumerate(features):
        if f in target:
            target_vector[0, i] = target[f] * weight_vector[i]
    
    # compute similarity
    similarity = cosine_similarity(target_vector, weighted_matrix)[0]
    
    top_indices = similarity.argsort()[::-1][:n]
    actual_indices = filtered_df.index[top_indices]
    
    results = df.loc[actual_indices][['track_name', 'artists', 'track_genre', 'popularity', 'valence', 'energy', 'mood']].copy()
    results['similarity'] = similarity[top_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# Experiment 1 - Equal Weights

In [3]:
weights_equal = {f: 1.0 for f in audio_features}
results = recommend_weighted('happy', df, df_audio_standard, audio_features, weights_equal)
print(results)

Mood: happy
Target → valence: 1.0, energy: 1.0
Weights: {'danceability': 1.0, 'energy': 1.0, 'mode': 1.0, 'speechiness': 1.0, 'acousticness': 1.0, 'instrumentalness': 1.0, 'liveness': 1.0, 'valence': 1.0, 'tempo': 1.0, 'duration_min': 1.0, 'explicit': 1.0}

Songs in mood pool: 32992
                                 track_name  \
1   Zurück zu dir (Hallo Klaus) - Party Mix   
2                          Laat Maar Waaien   
3                Nu har pappa laddat bössan   
4                              サイレントマジョリティー   
5                Zipfel eini , Zipfel aussi   
6                             Blue Blue Day   
7                             Winter's Gone   
8                    That's The Way Love Is   
9                    That's The Way Love Is   
10               Flower Garden(version2016)   

                                   artists      track_genre  popularity  \
1   Mike Der Bademeister;Die Spassrebellen            party          26   
2                               Lawineboys  happ

# Experiment 2 - Higher Weights for Valence and Energy

In [4]:
weights_mood = {
    'valence': 3.0,
    'energy': 3.0,
    'danceability': 2.0,
    'mode': 1.0,
    'speechiness': 0.5,
    'acousticness': 1.0,
    'instrumentalness': 0.5,
    'liveness': 0.5,
    'tempo': 1.0,
    'duration_min': 0.5,
    'explicit': 0.5
}

results = recommend_weighted('happy', df, df_audio_standard, audio_features, weights_mood)
print(results)

Mood: happy
Target → valence: 1.0, energy: 1.0
Weights: {'valence': 3.0, 'energy': 3.0, 'danceability': 2.0, 'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0, 'instrumentalness': 0.5, 'liveness': 0.5, 'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5}

Songs in mood pool: 32992
                                   track_name               artists  \
1                                サイレントマジョリティー          Keyakizaka46   
2                                サイレントマジョリティー          Keyakizaka46   
3                                 El Caterete          Wganda Kenya   
4               Jogi - I mog di so WM Version              voXXclub   
5                            Laat Maar Waaien            Lawineboys   
6   Bambälä (Die Seele wieder baumeln lassen)               DJ Ötzi   
7                                  Existência                 Pense   
8                                     500 mil               Mimikry   
9            Mama Laudaaa - Apres Ski Edition  Almklausi;Specktakel   
10    

# Experiment 3 - Mood Filter + Popularity Sort

In [5]:
def recommend_weighted_v2(mood, df, scaled_df, features, weights, n=10, genre=None):
    mood_targets = {
        'happy': {'valence': 1.0, 'energy': 1.0},
        'angry': {'valence': 0.0, 'energy': 1.0},
        'sad':   {'valence': 0.0, 'energy': 0.0},
        'calm':  {'valence': 1.0, 'energy': 0.0},
    }
    
    if mood not in mood_targets:
        print(f'Invalid mood. Choose from: {list(mood_targets.keys())}')
        return None
    
    print(f'Mood: {mood}')
    print(f'Weights: {weights}')
    print()
    
    # filter by mood
    filtered_df = df[df['mood'] == mood].copy()
    filtered_scaled = scaled_df.loc[filtered_df.index]
    
    # apply genre filter if specified
    if genre:
        genre_mask = filtered_df['track_genre'].str.contains(genre, case=False)
        filtered_df = filtered_df[genre_mask]
        filtered_scaled = filtered_scaled[genre_mask]
    
    print(f'Songs in mood pool: {len(filtered_df)}')
    
    # weighted similarity
    weight_vector = np.array([weights.get(f, 1.0) for f in features])
    weighted_matrix = filtered_scaled[features].values * weight_vector
    
    target = mood_targets[mood]
    target_vector = np.zeros((1, len(features)))
    for i, f in enumerate(features):
        if f in target:
            target_vector[0, i] = target[f] * weight_vector[i]
    
    similarity = cosine_similarity(target_vector, weighted_matrix)[0]
    filtered_df = filtered_df.copy()
    filtered_df['similarity'] = similarity
    
    # combined score: similarity + popularity
    filtered_df['popularity_norm'] = filtered_df['popularity'] / 100
    filtered_df['combined_score'] = (
        0.5 * filtered_df['similarity'] + 
        0.5 * filtered_df['popularity_norm']
    )
    
    results = filtered_df.nlargest(n, 'combined_score')[
        ['track_name', 'artists', 'track_genre', 'popularity', 
         'valence', 'energy', 'mood', 'similarity', 'combined_score']
    ].copy()
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

weights_mood = {
    'valence': 3.0, 'energy': 3.0, 'danceability': 2.0,
    'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0,
    'instrumentalness': 0.5, 'liveness': 0.5,
    'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5
}

results = recommend_weighted_v2('happy', df, df_audio_standard, audio_features, weights_mood)
print(results)

Mood: happy
Weights: {'valence': 3.0, 'energy': 3.0, 'danceability': 2.0, 'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0, 'instrumentalness': 0.5, 'liveness': 0.5, 'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5}

Songs in mood pool: 32992
                      track_name                artists  \
1                     After LIKE                    IVE   
2             Livin' On A Prayer               Bon Jovi   
3                I Ain't Worried            OneRepublic   
4   She Had Me At Heads Carolina          Cole Swindell   
5       You Give Love A Bad Name               Bon Jovi   
6                     The Nights                 Avicii   
7                     Can't Stop  Red Hot Chili Peppers   
8                     Take on Me                   a-ha   
9                     SUPERMODEL               Måneskin   
10       Smells Like Teen Spirit                Nirvana   

                            track_genre  popularity  valence  energy   mood  \
1                     

# Experiment 4 - All Moods

In [6]:
for mood in ['happy', 'angry', 'sad', 'calm']:
    print(f'\n{"="*50}')
    print(f'Mood: {mood.upper()}')
    print('='*50)
    results = recommend_weighted_v2(mood, df, df_audio_standard, audio_features, weights_mood)
    if results is not None:
        print(results[['track_name', 'artists', 'popularity', 'valence', 'energy', 'combined_score']])


Mood: HAPPY
Mood: happy
Weights: {'valence': 3.0, 'energy': 3.0, 'danceability': 2.0, 'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0, 'instrumentalness': 0.5, 'liveness': 0.5, 'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5}

Songs in mood pool: 32992
                      track_name                artists  popularity  valence  \
1                     After LIKE                    IVE          88    0.799   
2             Livin' On A Prayer               Bon Jovi          85    0.795   
3                I Ain't Worried            OneRepublic          96    0.825   
4   She Had Me At Heads Carolina          Cole Swindell          82    0.722   
5       You Give Love A Bad Name               Bon Jovi          82    0.812   
6                     The Nights                 Avicii          86    0.654   
7                     Can't Stop  Red Hot Chili Peppers          82    0.875   
8                     Take on Me                   a-ha          85    0.876   
9                 

# Experiment 5 - Different Weight Combinations

In [7]:
weight_combinations = {
    'equal_weights': {f: 1.0 for f in audio_features},
    'mood_focused': {'valence': 3.0, 'energy': 3.0, 'danceability': 2.0,
                     'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0,
                     'instrumentalness': 0.5, 'liveness': 0.5,
                     'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5},
    'valence_heavy': {'valence': 5.0, 'energy': 2.0, 'danceability': 1.0,
                      'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0,
                      'instrumentalness': 0.5, 'liveness': 0.5,
                      'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5},
    'energy_heavy': {'valence': 2.0, 'energy': 5.0, 'danceability': 1.0,
                     'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0,
                     'instrumentalness': 0.5, 'liveness': 0.5,
                     'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5},
}

for name, weights in weight_combinations.items():
    print(f'\n{"="*40}')
    print(f'Weights: {name} | Mood: happy')
    print('='*40)
    results = recommend_weighted_v2('happy', df, df_audio_standard, audio_features, weights)
    if results is not None:
        print(results[['track_name', 'artists', 'popularity', 'valence', 'energy']])


Weights: equal_weights | Mood: happy
Mood: happy
Weights: {'danceability': 1.0, 'energy': 1.0, 'mode': 1.0, 'speechiness': 1.0, 'acousticness': 1.0, 'instrumentalness': 1.0, 'liveness': 1.0, 'valence': 1.0, 'tempo': 1.0, 'duration_min': 1.0, 'explicit': 1.0}

Songs in mood pool: 32992
                               track_name                      artists  \
1                              After LIKE                          IVE   
2                      Livin' On A Prayer                     Bon Jovi   
3                               September           Earth, Wind & Fire   
4           Start Me Up - Remastered 2009           The Rolling Stones   
5                 Never Gonna Give You Up                  Rick Astley   
6   Bad Decisions (with BTS & Snoop Dogg)  benny blanco;BTS;Snoop Dogg   
7              Feel So Close - Radio Edit                Calvin Harris   
8                               Angeleyes                         ABBA   
9                              Can't Stop      

# Save Best Model

In [8]:
best_config = {
    'model': 'weighted_similarity_v2',
    'features': 'audio_features',
    'scaling': 'StandardScaler',
    'best_weights': {
        'valence': 3.0, 'energy': 3.0, 'danceability': 2.0,
        'mode': 1.0, 'speechiness': 0.5, 'acousticness': 1.0,
        'instrumentalness': 0.5, 'liveness': 0.5,
        'tempo': 1.0, 'duration_min': 0.5, 'explicit': 0.5
    },
    'combined_score': {
        'similarity_weight': 0.5,
        'popularity_weight': 0.5
    },
    'mood_thresholds': {
        'version': 'v1',
        'threshold': 0.5
    }
}

feature_matrix = df_audio_standard[audio_features].values
os.makedirs('../models/content_based', exist_ok=True)

np.save('../models/content_based/weighted_feature_matrix.npy', feature_matrix)

with open('../models/content_based/weighted_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

df[['track_id', 'track_name', 'artists', 'track_genre', 
    'popularity', 'valence', 'energy', 'mood']].to_csv(
    '../models/content_based/weighted_song_index.csv', index=True)

print('saved files:')
print('1. models/content_based/weighted_feature_matrix.npy')
print('2. models/content_based/weighted_config.json')
print('3. models/content_based/weighted_song_index.csv')

saved files:
1. models/content_based/weighted_feature_matrix.npy
2. models/content_based/weighted_config.json
3. models/content_based/weighted_song_index.csv


# Conclusion

In [9]:
print('='*50)
print('WEIGHTED SIMILARITY MODEL - CONCLUSION')
print('='*50)

print('\n--- Experiments Summary ---')
print('Exp 1: equal weights                → low popularity songs dominated')
print('Exp 2: valence/energy weighted      → still low popularity songs')
print('Exp 3: mood filter + combined score → best results, high popularity')
print('Exp 4: all 4 moods tested           → all working correctly')
print('Exp 5: different weight combos      → valence/energy heavy both good')

print('\n--- Best Configuration ---')
print('Features       → audio_features (11)')
print('Scaling        → StandardScaler')
print('Weights        → valence 3x, energy 3x, danceability 2x')
print('Combined score → 50% similarity + 50% popularity')
print('Mood threshold → v1 (0.5)')

print('\n--- Key Findings ---')
print('1. equal weights give low popularity results')
print('2. combined score (similarity + popularity) essential')
print('3. valence and energy are most important mood features')
print('4. mood filter before similarity calculation is critical')
print('5. all 4 moods produce musically relevant results')
print('6. valence_heavy → most positive songs')
print('7. energy_heavy → most intense, highest popularity songs')

print('\n--- Mood Mapping ---')
print('happy → high valence + high energy')
print('angry → low valence + high energy')
print('sad   → low valence + low energy')
print('calm  → high valence + low energy')

print('\n--- Limitations ---')
print('1. mood categories are simplified (only 4 moods)')
print('2. threshold 0.5 may misclassify borderline songs')
print('3. popularity bias may miss good niche mood songs')

print('\n--- Best Use Case ---')
print('mood based recommendation')
print('works well for: recommend happy/sad/calm/angry songs')

print('\n--- Saved Files ---')
print('models/content_based/weighted_feature_matrix.npy')
print('models/content_based/weighted_config.json')
print('models/content_based/weighted_song_index.csv')

WEIGHTED SIMILARITY MODEL - CONCLUSION

--- Experiments Summary ---
Exp 1: equal weights                → low popularity songs dominated
Exp 2: valence/energy weighted      → still low popularity songs
Exp 3: mood filter + combined score → best results, high popularity
Exp 4: all 4 moods tested           → all working correctly
Exp 5: different weight combos      → valence/energy heavy both good

--- Best Configuration ---
Features       → audio_features (11)
Scaling        → StandardScaler
Weights        → valence 3x, energy 3x, danceability 2x
Combined score → 50% similarity + 50% popularity
Mood threshold → v1 (0.5)

--- Key Findings ---
1. equal weights give low popularity results
2. combined score (similarity + popularity) essential
3. valence and energy are most important mood features
4. mood filter before similarity calculation is critical
5. all 4 moods produce musically relevant results
6. valence_heavy → most positive songs
7. energy_heavy → most intense, highest popularity 